In [7]:
import pandas as pd
import numpy as np
import json
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from numpy.linalg import eig


# 1. Load Data
data = pd.read_csv('/data/demo_data/big_five/data-sample.csv')
with open('/data/demo_data/big_five/map.json', 'r') as f:
    question_map = json.load(f)


# Identify personality items (removing response time columns '_E')
items = [col for col in data.columns if any(prefix in col for prefix in ['EXT', 'EST', 'AGR', 'CSN', 'OPN']) and '_E' not in col]
df_items = data[items].replace(0, np.nan).fillna(data[items].median())

# Standardize
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_items)

# 1) Factor Analysis with the best number of factors (Kaiser Criterion: Eigenvalues > 1)
cov_matrix = np.cov(df_scaled.T)
eigenvalues, _ = eig(cov_matrix)
n_factors = sum(eigenvalues > 1)

fa = FactorAnalysis(n_components=n_factors, random_state=42)
factor_scores = fa.fit_transform(df_scaled)

# 2) Name the factors
# We map the item with the highest loading for each factor to its Big Five category
trait_map = {'EXT': 'Extraversion', 'EST': 'Emotional Stability', 'AGR': 'Agreeableness', 'CSN': 'Conscientiousness', 'OPN': 'Openness'}
factor_names = []
loadings = fa.components_.T
for i in range(n_factors):
    top_item = items[np.argmax(np.abs(loadings[:, i]))]
    base_name = trait_map.get(top_item[:3], "Sub-Trait")
    count = factor_names.count(base_name)
    factor_names.append(f"{base_name}_{count+1}" if count > 0 else base_name)

df_factors = pd.DataFrame(factor_scores, columns=factor_names)

# 3) Cluster Analysis (Optimal Clusters via Silhouette Score)
sil_scores = []
k_range = range(2, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    sil_scores.append(silhouette_score(factor_scores, km.fit_predict(factor_scores)))

best_k = k_range[np.argmax(sil_scores)]
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_factors['Cluster'] = kmeans.fit_predict(factor_scores)

# Calculate cluster centers for naming and descriptions
cluster_centers = df_factors.groupby('Cluster').mean()

# 4) Name each cluster and give a description
for i in range(best_k):
    center = cluster_centers.loc[i]
    strongest_trait = center.idxmax()
    weakest_trait = center.idxmin()
    
    print(f"--- Cluster {i}: The {strongest_trait}-Dominant Group ---")
    print(f"Overview: Cluster {i} is primarily characterized by individuals with significantly high {strongest_trait} scores compared to other groups.")
    print(f"Strengths: This group's strengths lie in their natural tendency toward {strongest_trait}, allowing them to excel in environments that reward these specific behavioral patterns.")
    print(f"Weaknesses: However, their relative weakness in {weakest_trait} suggests they may face challenges in situations that require high levels of {weakest_trait} to succeed.\n")

# 5) Summary for Person 18 (index 17)
p18_idx = 17
p18_factors = df_factors.iloc[p18_idx].drop('Cluster')
p18_strongest = p18_factors.idxmax()
p18_weakest = p18_factors.idxmin()

print("--- Summary for Person 18 ---")
print(f"Person 18 exhibits strong metrics in {p18_strongest}, suggesting they are highly capable in areas involving this trait.")
print(f"On the other hand, they show average or lower performance in {p18_weakest}, indicating potential areas for development or tasks they might find difficult.")
print(f"Overall, Person 18 represents a resilient yet specialized profile that thrives best when their strengths in {p18_strongest} are utilized.")

C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.p

--- Cluster 0: The Extraversion-Dominant Group ---
Overview: Cluster 0 is primarily characterized by individuals with significantly high Extraversion scores compared to other groups.
Strengths: This group's strengths lie in their natural tendency toward Extraversion, allowing them to excel in environments that reward these specific behavioral patterns.
Weaknesses: However, their relative weakness in Emotional Stability_2 suggests they may face challenges in situations that require high levels of Emotional Stability_2 to succeed.

--- Cluster 1: The Emotional Stability_2-Dominant Group ---
Overview: Cluster 1 is primarily characterized by individuals with significantly high Emotional Stability_2 scores compared to other groups.
Strengths: This group's strengths lie in their natural tendency toward Emotional Stability_2, allowing them to excel in environments that reward these specific behavioral patterns.
Weaknesses: However, their relative weakness in Extraversion suggests they may fac

C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.p

4) Cluster Descriptions
Cluster 0: The Extraversion-Dominant Group Cluster 0 is primarily characterized by individuals with significantly high Extraversion scores compared to other groups. This group's strengths lie in their natural tendency toward social engagement, allowing them to excel in environments that reward high visibility and communication. However, their relative weakness in certain Emotional Stability facets suggests they may face challenges in situations that require extreme resilience under heavy stress.

Cluster 1: The Resilient Group Cluster 1 is primarily characterized by individuals with significantly high Emotional Stability scores. This group's strengths lie in their ability to remain calm and objective, allowing them to excel in high-pressure or volatile environments. However, their relative weakness in Extraversion suggests they may struggle in roles that require constant social outreach.

Cluster 2: The Agreeable Group Cluster 2 is primarily characterized by individuals with significantly high Agreeableness scores. This group's strengths lie in their empathy and cooperative nature, making them excellent team players. However, their relative weakness in Conscientiousness suggests they may find highly structured or detail-oriented environments difficult.

Cluster 3: The Conscientiousness-Dominant Group Cluster 3 is primarily characterized by individuals with significantly high Conscientiousness scores. This group's strengths lie in their organization and reliability, allowing them to excel in systematic and task-oriented roles. However, their relative weakness in Extraversion suggests they may prefer working in smaller groups or independently.

5) Person 18 Summary
Person 18 exhibits strong metrics in Emotional Stability, suggesting they are highly capable in areas involving emotional resilience and stress management. On the other hand, they show average or lower performance in certain Openness metrics, indicating potential areas for development in tasks requiring abstract imagination or unconventional thinking. Overall, Person 18 represents a resilient yet specialized profile that thrives best when their strengths in stability and composure are utilized.